In [1]:
!nvidia-smi

Thu Jun 19 12:09:30 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   45C    P0             453W / 700W |  59979MiB / 81559MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
from parser import (remove_comments_and_docstrings,
                   tree_to_token_index,
                   index_to_code_token,
                   tree_to_variable_index)
from tree_sitter import Language, Parser

In [3]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [4]:
from preprocess import AST

from unixcoder import UniXcoder
import torch
device = torch.device("cuda:0")

/home/jovyan/miniconda3/envs/kamikazi-llava/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = UniXcoder("microsoft/unixcoder-base").to(device)

In [6]:
model

UniXcoder(
  (model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(51416, 768, padding_idx=1)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
      (token_type_embeddings): Embedding(10, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [7]:
# Encode maximum function
func_max = "def f(a,b): if a>b: return a else return b"
tokens_ids = model.tokenize([func_max],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,max_func_embedding = model(source_ids)

# Encode minimum function
func_min = "def f(a,b): if a<b: return a else return b"
tokens_ids = model.tokenize([func_min],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,min_func_embedding = model(source_ids)

# Encode NL
nl = "return maximum value"
tokens_ids = model.tokenize([nl],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,nl_embedding = model(source_ids)

print(max_func_embedding.shape)
print(max_func_embedding)

torch.Size([1, 768])
tensor([[ 8.6533e-01, -1.9796e+00, -8.6848e-01,  4.2653e-01, -5.3695e-01,
         -1.5521e-01,  5.3771e-01,  3.4199e-01,  3.6306e-01, -3.9391e-01,
         -1.1816e+00,  2.6010e+00, -7.7132e-01,  1.8441e+00,  2.3645e+00,
         -8.0971e-01,  1.9955e+00,  6.4072e-01, -2.4817e-01, -2.6337e+00,
         -1.1198e+00, -2.3594e+00, -7.5278e-01,  1.1628e-01,  1.6286e+00,
         -4.5870e-02, -1.4457e+00,  7.8070e-01,  7.7836e-01, -5.6095e-01,
          2.2112e+00, -7.8929e-01, -2.3475e+00,  1.7342e+00, -8.8963e-01,
         -6.9368e-01,  2.4841e+00,  4.9074e-01, -8.5598e-01,  1.0941e-01,
         -1.4604e+00,  2.3402e+00, -1.8161e+00, -2.9290e+00, -3.9770e+00,
         -4.7137e-01,  1.7589e+00,  2.5646e+00, -4.2621e-01, -8.0808e-02,
         -2.9151e+00,  1.6699e+00, -1.9765e+00, -8.8961e-01,  8.9094e-01,
         -6.7859e-01, -4.2709e-01, -2.1378e+00, -2.3407e+00, -4.9258e-01,
          1.6661e+00, -1.4553e+00, -4.7143e-01,  1.7414e+00, -1.4021e+00,
         -3.6194e

In [8]:
source_ids.ne(model.config.pad_token_id)

tensor([[True, True, True, True, True, True, True]], device='cuda:0')

In [9]:
source_ids.shape

torch.Size([1, 7])

In [10]:
# Normalize embedding
norm_max_func_embedding = torch.nn.functional.normalize(max_func_embedding, p=2, dim=1)
norm_min_func_embedding = torch.nn.functional.normalize(min_func_embedding, p=2, dim=1)
norm_nl_embedding = torch.nn.functional.normalize(nl_embedding, p=2, dim=1)

max_func_nl_similarity = torch.einsum("ac,bc->ab",norm_max_func_embedding,norm_nl_embedding)
min_func_nl_similarity = torch.einsum("ac,bc->ab",norm_min_func_embedding,norm_nl_embedding)
max_func_min_func_similarity = torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_min_func_embedding)

print(max_func_nl_similarity)
print(min_func_nl_similarity)
print(max_func_min_func_similarity)

tensor([[0.3002]], device='cuda:0', grad_fn=<ViewBackward0>)
tensor([[0.1881]], device='cuda:0', grad_fn=<ViewBackward0>)
tensor([[0.8428]], device='cuda:0', grad_fn=<ViewBackward0>)


In [11]:
model.tokenize(['def func(a):'], mode="<encoder-only>")

[[0, 6, 2, 729, 2666, 126, 183, 953, 2]]

In [12]:
model.tokenizer.decode([0, 6 ,2, 729, 2666, 126])

'<s><encoder-only></s>def func('

In [13]:
model.tokenizer.decode([0, 2666, 2])

'<s> func</s>'

In [14]:
model.tokenizer.tokenize('def func(a):')

['def', 'Ġfunc', '(', 'a', '):']

In [15]:
min_func_embedding.shape, tokens_embeddings.shape

(torch.Size([1, 768]), torch.Size([1, 7, 768]))

In [16]:
max_length = 512
max_ast = AST(func_max, 'python', model.tokenizer) # add spaces? 
tokens = max_ast[:max_length-4]
tokens = [model.tokenizer.cls_token, "<encoder-only>", model.tokenizer.sep_token] + tokens + [model.tokenizer.sep_token]
tokens_ids = model.tokenizer.convert_tokens_to_ids(tokens)
source_ids = torch.tensor(tokens_ids).unsqueeze(0).to(device)
tokens_embeddings, max_func_ast_embedding = model(source_ids)

max_length = 512
min_ast = AST(func_min, 'python', model.tokenizer) # add spaces? 
tokens = min_ast[:max_length-4]
tokens = [model.tokenizer.cls_token, "<encoder-only>", model.tokenizer.sep_token] + tokens + [model.tokenizer.sep_token]
tokens_ids = model.tokenizer.convert_tokens_to_ids(tokens)
source_ids = torch.tensor(tokens_ids).unsqueeze(0).to(device)
tokens_embeddings, min_func_ast_embedding = model(source_ids)

In [26]:
max_func_ast_embedding.shape

torch.Size([1, 768])

In [17]:
source_ids

tensor([[    0,     6,     2, 50973,   729,   188, 50513,   126,   183,   130,
           184,   127, 50512,   144, 50589, 51243,   406,   183,   146,   184,
         51242,   144,   427,   183,   613, 50588, 51113,   427,   184, 51112,
         50972,     2]], device='cuda:0')

In [18]:
model.tokenizer.decode([729, 763, 50513])

'deffuncAST#parameters#Left'

In [19]:
tokens

['<s>',
 '<encoder-only>',
 '</s>',
 'AST#function_definition#Left',
 'def',
 'f',
 'AST#parameters#Left',
 '(',
 'a',
 ',',
 'b',
 ')',
 'AST#parameters#Right',
 ':',
 'AST#ERROR#Left',
 'AST#comparison_operator#Left',
 'if',
 'a',
 '<',
 'b',
 'AST#comparison_operator#Right',
 ':',
 'return',
 'a',
 'else',
 'AST#ERROR#Right',
 'AST#return_statement#Left',
 'return',
 'b',
 'AST#return_statement#Right',
 'AST#function_definition#Right',
 '</s>']

In [20]:
tokens_ids

[0,
 6,
 2,
 50973,
 729,
 188,
 50513,
 126,
 183,
 130,
 184,
 127,
 50512,
 144,
 50589,
 51243,
 406,
 183,
 146,
 184,
 51242,
 144,
 427,
 183,
 613,
 50588,
 51113,
 427,
 184,
 51112,
 50972,
 2]

In [21]:
for key in model.tokenizer.get_vocab():
    # if 'ast' in key.lower():
    if 'AST#function_definition#Left' in key:
        print(key)

AST#function_definition#Left


In [22]:
model.tokenizer.decode([0, 729, 2, 0, 188, 2, 0, 126, 2])

'<s>def</s><s>f</s><s>(</s>'

In [23]:
# Normalize embedding
norm_max_func_embedding = torch.nn.functional.normalize(max_func_embedding, p=2, dim=1)
norm_min_func_embedding = torch.nn.functional.normalize(min_func_embedding, p=2, dim=1)
norm_max_func_ast_embedding = torch.nn.functional.normalize(max_func_ast_embedding, p=2, dim=1)
norm_min_func_ast_embedding = torch.nn.functional.normalize(min_func_ast_embedding, p=2, dim=1)
norm_nl_embedding = torch.nn.functional.normalize(nl_embedding, p=2, dim=1)

max_func_nl_similarity = torch.einsum("ac,bc->ab",norm_max_func_embedding,norm_nl_embedding)
min_func_nl_similarity = torch.einsum("ac,bc->ab",norm_min_func_embedding,norm_nl_embedding)

print(max_func_nl_similarity)
print(min_func_nl_similarity)

tensor([[0.3002]], device='cuda:0', grad_fn=<ViewBackward0>)
tensor([[0.1881]], device='cuda:0', grad_fn=<ViewBackward0>)


In [24]:
print('max func - max ast: {:.2f}, max func - min ast: {:.2f}, min func - max ast: {:.2f}, min func - min ast: {:.2f}, max ast - nl max: {:.2f}, min ast - nl max: {:.2f}'.format(
    torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_max_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_min_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_embedding, norm_max_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_embedding, norm_min_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_max_func_ast_embedding, norm_nl_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_ast_embedding, norm_nl_embedding).item()))

max func - max ast: 0.86, max func - min ast: 0.76, min func - max ast: 0.72, min func - min ast: 0.84, max ast - nl max: 0.25, min ast - nl max: 0.17


In [25]:
!nvidia-smi

Thu Jun 19 12:09:50 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   45C    P0             460W / 700W |  59979MiB / 81559MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--